In [0]:
dbutils.widgets.text("env", "dev", "env")
env = dbutils.widgets.get("env")
print(env)

In [0]:
import json
import requests
file_path = f"../config/{env}.json"
with open(file_path, "r") as f:
    config = json.load(f)


In [0]:
workspace = config['workspace']
metadata_schema = config['metadata']

In [0]:
source_mapping = {
    'github':config['github'],
    'master':config['master'],
    'finnhub':config['finnhub']
}

In [0]:
target_mapping = {
    'master':config['github'],
    'm101_finnhub':config['m101_finnhub']
}

In [0]:
execution_type = {
    "ingestion": "ni",
    "consumption": "con"
}

In [0]:
ingestion_type = {
    'finnhub':'url_ingestion',
    'ON_PREM':'RDBMS', #features added in future
    'SRC3':'FILE' #features added in future
}

In [0]:
# DBTITLE 1,Insert Log
def insert_log(param_name, process_name, source_schema, source_table, target_schema, target_table, process_date,process_status, log):
    # error_msg = str(log).replace("'", "''")
    spark.sql(
        f"""
        INSERT INTO {metadata_schema}.audit_log_{env} 
        VALUES ('{param_name}', '{process_name}', '{source_schema}', '{source_table}', '{target_schema}', '{target_table}', '{process_date}', current_timestamp(), '{process_status}', '{log}')
        """
    )

In [0]:
# DBTITLE 1,Get GitHub Events
def get_github_events(src):
    try:
        TOKEN = source_mapping[src]
        headers = {
            "Authorization": f"Bearer {TOKEN}",
            "Accept": "application/vnd.github+json"
        }
        url = "https://api.github.com/events"
        response = requests.get(url, headers=headers)
        return response.status_code
    except Exception as e:
        return f"Error: {str(e)}"

In [0]:
# function for getting finnhub data
def get_finnhub_connection(symbol):
    url = "https://finnhub.io/api/v1/quote"
    params = {
        "symbol": symbol,
        "token": config['finnhub']
    }
    response = requests.get(url, params=params)
    return response.json()

In [0]:
# print(get_finnhub_connection('AAPL', config['finnhub']))

In [0]:
#function for checking if target is already registered
def check_param_name_for_tgt(tgt):
    get_param_details = spark.sql(f"""select * from metadata_{env}.param_table_{env} where param_name='{tgt}'""").toPandas().to_dict('records')
    if len(get_param_details) == 0:
        return False
    else:
        return True



In [0]:
#function for registering target
def register_tgt(tgt, execution_type):
    if not check_param_name_for_tgt(tgt):
        spark.sql(f"""insert into metadata_{env}.param_table_{env} values ('{tgt}', 'Y', cast(date_format(current_date(), 'yyyyMMdd') as int),'{execution_type}')""")
        spark.sql(f"""create schema if not exists {workspace}.{target_mapping[tgt]} """)
        return f"target registered successfully"
    else:
        return f"target already registered"


In [0]:
#function for getting source schema
def f_get_src_schema(src):
    if src in config:
        return config[src]
    else:
        return f"source not found : "+src

In [0]:
#function for getting target schema
def f_get_tgt_schema(tgt):
    if tgt in config:
        return config[tgt]
    else:
        return f"source not found : "+tgt

In [0]:

#getting records from dp_metadata for each process

def get_process_details(param_name,process_name,tgt_table_list):
    tgt_tbl = tgt_table_list[0]['target_table']
    try:
        df_process_details = spark.sql(
            f"select * from {metadata_schema}.dp_metadata_{env} where param_name='{param_name}' and process_name  = '{process_name}' and target_table = '{tgt_tbl}'"
        )
        records = df_process_details.toPandas().to_dict('records')
        if not records:
            print(f"No records found in  dp_metadata for process_name: {process_name} and target_table: {tgt_tbl}")
            return False
        return records
    except Exception as e:
        print(f"Error fetching process details: {e}")
        return e


In [0]:
#getting records from table_column_metadata for each target_table
def get_table_column_details(tgt_table_list):
    tgt_tbl = tgt_table_list[0]['target_table']
    tgt_schema = tgt_table_list[0]['target_schema']
    try:
        df_process_details = spark.sql(
            f"select * from {metadata_schema}.table_column_metadata_{env} where target_schema='{tgt_schema}' and  target_table = '{tgt_tbl}'"
        )
        records = df_process_details.toPandas().to_dict('records')
        if not records:
            print(f"No records found in  table_column_metadata for target_schema: {tgt_schema} and target_table: {tgt_tbl}")
            return False
        return records
    except Exception as e:
        print(f"Error fetching process details: {e}")
        return e

In [0]:
def insert_process_details(src_tbl_list, tgt_tbl_list, param_name, process_name):
    try:
        for i in range(len(src_tbl_list)):
            spark.sql(
                f"""insert into {metadata_schema}.dp_metadata_{env} values ('{param_name}','{process_name}','{src_tbl_list[i]["source_schema"]}','{src_tbl_list[i]["source_table"]}','{tgt_tbl_list[0]["target_schema"]}','{tgt_tbl_list[0]["target_table"]}','{tgt_tbl_list[0]["frequency"]}','{tgt_tbl_list[0]["holiday_ind"]}','{tgt_tbl_list[0]["weekend_ind"]}','Y')"""
            )
        return f"process details inserted successfully"
    except Exception as e:
        print(f"Error inserting process details: {e}")
        return f"error  Error inserting process details: {e}"

In [0]:
def insert_table_column_details(src_tgt_mapping_list, process_date):
    try:
        for i in range(len(src_tgt_mapping_list)):
            spark.sql(
                f"""insert into {metadata_schema}.table_column_metadata_{env} values ('{src_tgt_mapping_list[i]["source_schema"]}','{src_tgt_mapping_list[i]["source_table"]}','{src_tgt_mapping_list[i]["source_column_name"]}','{src_tgt_mapping_list[i]["source_column_datatype"]}','{src_tgt_mapping_list[i]["target_schema"]}','{src_tgt_mapping_list[i]["target_table"]}','{src_tgt_mapping_list[i]["target_column_name"]}','{src_tgt_mapping_list[i]["target_column_datatype"]}','Y',{process_date})"""
            )
        return "table column details inserted successfully"
    except Exception as e:
        print(f"Error inserting table column details: {e}")
        return f"error Error inserting table column details: {e}"

In [0]:
def run_load_metadata(src_tbl_list,tgt_tbl_list,param_name,process_name,src_tgt_mapping_list,process_date):
    if get_process_details(param_name,process_name,tgt_tbl_list):
        print("process already exists")
        metadata_insert_process_status = "success"
        log = "process already exists"
    else:
        print("process does not exists")
        comment = insert_process_details(src_tbl_list,tgt_tbl_list,param_name,process_name)
        log = str(comment).replace("'", "''")
        if log == "process details inserted successfully":
            metadata_insert_process_status = "success"
        else:
            metadata_insert_process_status = "failed"
    insert_log(param_name, process_name, "insert_metadata_process", "insert_metadata_process", "insert_metadata_process", "insert_metadata_process",process_date,metadata_insert_process_status,log)
    
    if get_table_column_details(tgt_tbl_list):
        print("table_column_metadata already exists")
        table_column_metadata_process_status = "success"
        log = "table_column_metadata already exists"
    else:
        print("table_column_metadata does not exists")
        # insert_table_column_details
        comment = insert_table_column_details(src_tgt_mapping_list,process_date)
        log = str(comment).replace("'", "''")
        if log == "table column details inserted successfully":
            table_column_metadata_process_status = "success"
        else:
            table_column_metadata_process_status = "failed"
    insert_log(param_name, process_name, "insert_table_column_metadata_process", "insert_table_column_metadata_process", "insert_table_column_metadata_process", "insert_table_column_metadata_process",process_date,table_column_metadata_process_status,log)
    if metadata_insert_process_status == "success"  and table_column_metadata_process_status == "success":
        return "success"
    else:
        return "failed"
    
    

In [0]:
def fetch_column_details(target_schema,target_table):
    all_columns_list = spark.sql(f"""select * from {metadata_schema}.table_column_metadata_{env} where target_schema='{target_schema}' and target_table = '{target_table}'""").toPandas().to_dict('records')
    # print(all_columns_list)
    return all_columns_list


In [0]:
def create_tgt_tbl(tgt_tbl_list):
    tgt_table_schema = tgt_tbl_list[0]['target_schema']
    tgt_table_name = tgt_tbl_list[0]['target_table']
    partition_column  = "process_date"
    all_columns_list = fetch_column_details(tgt_table_schema,tgt_table_name)
    column_query = ""
    for i in all_columns_list:
        column_name = i['target_column_name']
        column_datatype = i['target_column_datatype']
        pre_query = column_name + " " + column_datatype + ","
        column_query += pre_query
    print(column_query)

    hash_id = "hash_id"
    try:
        spark.sql(f"create database if not exists {tgt_table_schema}")
        query = f"""create table if not exists {tgt_table_schema}.{tgt_table_name} ({hash_id} string not null,{column_query} r_source string not null,{partition_column} int not null ) USING DELTA partitioned by ({partition_column})"""
        print(query)
        spark.sql(query)
        return f"Table {tgt_table_schema}.{tgt_table_name} created successfully"
    except Exception as e:
        return f"Error creating table: {e}"
        


In [0]:
import hashlib

def create_hash_id(row_dict):
    # Sort keys to ensure consistent hash regardless of key order
    sorted_items = sorted(row_dict.items())
    hash_input = '|'.join(f"{k}:{str(v)}" for k, v in sorted_items)
    # print(hash_input)
    hash_id = hashlib.md5(hash_input.encode('utf-8')).hexdigest()
    return hash_id

In [0]:
def rollback_run(tgt_tbl_list,process_date):
    tgt_table_schema = tgt_tbl_list[0]['target_schema']
    tgt_table_name = tgt_tbl_list[0]['target_table']
    query =f"""select max(version) from (DESCRIBE HISTORY  {tgt_table_schema}.{tgt_table_name}) where CAST(date_format(timestamp, 'yyyyMMdd') AS INT)= (SELECT max(CAST(date_format(timestamp, 'yyyyMMdd') AS INT)) FROM (DESCRIBE HISTORY  {tgt_table_schema}.{tgt_table_name}) where CAST(date_format(timestamp, 'yyyyMMdd') AS INT) < {process_date}) """
    print(query)
    prev_version_df = spark.sql(query)
    prev_version = prev_version_df.collect()[0][0] if prev_version_df.count() > 0 else None
    if prev_version is not None:
        spark.sql(f"RESTORE TABLE {tgt_table_schema}.{tgt_table_name} TO VERSION AS OF {prev_version}")
        return f"Table {tgt_table_schema}.{tgt_table_name} restored to version {prev_version}"
    else:
        return f"No previous version found for {tgt_table_schema}.{tgt_table_name}"
    